# 03 — Clean Proximity Data
Loads ECHO PFAS facilities, PWS service area boundaries, PFAS measurements, and demographics. Merges them into a single analysis-ready dataset saved to `data/cleaned_data/`.

**Inputs:** `data/01_pulls/`, `data/cleaned_data/final_clean_census.csv`  
**Outputs:** `data/cleaned_data/proximity_final.geojson`, `data/cleaned_data/proximity_final.csv`, `data/cleaned_data/echo_facilities_with_pws.csv`

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

# ── Helper functions ──────────────────────────────────────────────────────────
def find_repo_root(start=Path().resolve()):
    """Search upward for the repo root (directory containing README.md + data/)."""
    for parent in [start] + list(start.parents):
        if (parent / 'README.md').exists() and (parent / 'data').exists():
            return parent
    raise FileNotFoundError('Could not find repo root.')

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
DATA_IN  = REPO_ROOT / 'data' / '01_pulls'
DATA_OUT = REPO_ROOT / 'data' / 'cleaned_data'
DATA_OUT.mkdir(exist_ok=True)

print('Working directory:', os.getcwd())
print('Input:  ', DATA_IN)
print('Output: ', DATA_OUT)

## Loading Raw Data
Loading the files saved by `01_data_pull_me_search.ipynb` from `data/01_pulls/`.

In [14]:
# Load ECHO PFAS-handling facilities
echo_df = pd.read_csv(
    DATA_IN / 'echo_pfas_facilities_nj.csv',
    dtype={'FAC_FIPS_CODE': str}
)
echo_df['Latitude']  = pd.to_numeric(echo_df['Latitude'],  errors='coerce')
echo_df['Longitude'] = pd.to_numeric(echo_df['Longitude'], errors='coerce')
echo_df = echo_df.dropna(subset=['Latitude', 'Longitude'])

print(f'ECHO facilities loaded: {len(echo_df)}')
print(echo_df[['Facility', 'Industry', 'Latitude', 'Longitude']].head())

ECHO facilities loaded: 3757
                Facility  Industry   Latitude  Longitude
0  FLY-N-D LANDING STRIP  Airports  40.468437 -75.002391
1             WEISS FARM  Airports  40.923700 -74.870400
2       ALLOWAY AIRFIELD  Airports  39.541800 -75.304400
3                 TRINCA  Airports  40.966760 -74.780170
4       AEROFLEX-ANDOVER  Airports  41.008600 -74.738000


In [15]:
# Load PWS service area boundaries
pws_gdf = gpd.read_file(DATA_IN / 'pws_boundaries_nj.geojson')

print(f'PWS boundaries loaded: {len(pws_gdf)}')
print(pws_gdf[['PWSID', 'PWS_Name', 'Population_Served_Count']].head())

PWS boundaries loaded: 558
       PWSID                  PWS_Name  Population_Served_Count
0  NJ0107001      NJAW EGG HARBOR CITY                   4900.0
1  NJ0436008           WINSLOW TWP DMU                    172.0
2  NJ0111007          SWAN LAKE RESORT                     70.0
3  NJ1714001      COUNTRY CLUB ESTATES                    300.0
4  NJ1530005  STAFFORD TWP MUA FAWN LA                   1740.0


In [16]:
# Load PWS → Census tract crosswalk
xwalk_df = pd.read_csv(
    DATA_IN / 'pws_tract_crosswalk_nj.csv',
    dtype={'GEOID20': str, 'PWSID': str}
)

print(f'Tract crosswalk loaded: {len(xwalk_df)} rows')
print(xwalk_df.head())

Tract crosswalk loaded: 3833 rows
       GEOID20      PWSID  Tract_Km  Tract_I_Km  Area_Weight  Pop20_AW  \
0  34001000200  NJ0102001  0.523371    0.487665     0.931777      2699   
1  34001000200  NJ0122001  0.523371    0.000003     0.000006         0   
2  34001000300  NJ0102001  0.365553    0.341909     0.935319      3767   
3  34001000400  NJ0102001  0.676905    0.597791     0.883124      2864   
4  34001000500  NJ0102001  0.273704    0.270606     0.988680      2978   

   Tract_Buildings  Tract_O_Buildings  Bldg_Weight  Pop20_BW  
0            597.0                596     0.998325    2892.0  
1            597.0                  0     0.000000       0.0  
2            404.0                403     0.997525    4017.0  
3            276.0                270     0.978261    3172.0  
4            394.0                394     1.000000    3012.0  


In [17]:
# Load UCMR5 PFAS data and aggregate by PWSID
#Very similar code to aggreating by Zipcode, but again differs
ucmr_df = pd.read_csv(DATA_IN / 'ucmr5_nj_slim.csv', dtype={'PWSID': str})
ucmr_df['AnalyticalResultValue'] = ucmr_df['AnalyticalResultValue'].fillna(0)

# Keep only PFAS contaminants
pfas_df = ucmr_df[
    ucmr_df['Contaminant'].str.startswith('PF') |
    ucmr_df['Contaminant'].str.contains('HFPO-DA')
].copy()

# Aggregate to one row per PWSID
pfas_by_pws = pfas_df.groupby('PWSID').agg(
    max_pfas         = ('AnalyticalResultValue', 'max'),
    mean_pfas        = ('AnalyticalResultValue', 'mean'),
    total_detections = ('AnalyticalResultValue', lambda x: (x > 0).sum()),
    pws_name         = ('PWSName', 'first')
).reset_index()

pfas_by_pws['exceeds_mcl_max'] = pfas_by_pws['max_pfas'] > 0.004
pfas_by_pws['exceeds_mcl_mean'] = pfas_by_pws['mean_pfas'] > 0.004

print(pfas_by_pws.head())

PFAS data: 265 water systems
       PWSID  max_pfas  mean_pfas  total_detections  \
0  NJ0102001    0.0073   0.000501                 8   
1  NJ0102301    0.0000   0.000000                 0   
2  NJ0102302    0.0000   0.000000                 0   
3  NJ0103001    0.0071   0.000282                 6   
4  NJ0104003    0.0000   0.000000                 0   

                      pws_name  exceeds_mcl_max  exceeds_mcl_mean  
0            ATLANTIC CITY MUA             True             False  
1        RESORTS ATLANTIC CITY            False             False  
2     BALLYS PARK PLACE CASINO            False             False  
3  BRIGANTINE WATER DEPARTMENT             True             False  
4            BUENA BOROUGH MUA            False             False  


In [18]:
# Load cleaned census demographics (from notebook 02)
census_df = pd.read_csv(
    DATA_OUT / 'final_clean_census.csv',
    dtype={'ZIPCODE': str}
)

# Load ZIP → PWSID crosswalk to link demographics to water systems
zip_pws = pd.read_csv(
    DATA_IN / 'zipcodes_raw.csv',
    dtype={'PWSID': str, 'ZIPCODE': str}
)
zip_pws['ZIPCODE'] = zip_pws['ZIPCODE'].str.zfill(5)

print(f'Census data: {len(census_df)} ZIPs')
print(f'ZIP-PWS crosswalk: {len(zip_pws)} rows')

Census data: 598 ZIPs
ZIP-PWS crosswalk: 31107 rows


## Spatial Join: ECHO Facilities → PWS Service Areas

In [19]:
# Convert ECHO facilities to GeoDataFrame
echo_gdf = gpd.GeoDataFrame(
    echo_df,
    geometry = gpd.points_from_xy(echo_df['Longitude'], echo_df['Latitude']),
    crs      = 'EPSG:4326'
)

# Match CRS
pws_gdf = pws_gdf.to_crs('EPSG:4326')

# Spatial join: which PWS service area does each facility fall in?
facilities_in_pws = gpd.sjoin(
    echo_gdf,
    pws_gdf[['PWSID', 'PWS_Name', 'geometry']],
    how       = 'left',
    predicate = 'within'
)

matched = facilities_in_pws['PWSID'].notna().sum()
print(f'Facilities matched to a PWS service area: {matched} of {len(echo_gdf)}')
print(facilities_in_pws[['Facility', 'Industry', 'PWSID']].head())

Facilities matched to a PWS service area: 3258 of 3757
                Facility  Industry PWSID
0  FLY-N-D LANDING STRIP  Airports   NaN
1             WEISS FARM  Airports   NaN
2       ALLOWAY AIRFIELD  Airports   NaN
3                 TRINCA  Airports   NaN
4       AEROFLEX-ANDOVER  Airports   NaN


In [20]:
# Aggregate facilities per PWSID: total count and count by industry type
fac_matched = facilities_in_pws[facilities_in_pws['PWSID'].notna()].copy()

# Total facility count per PWS
fac_count = fac_matched.groupby('PWSID').agg(
    total_facilities = ('Facility', 'count'),
).reset_index()

# Count by industry type (pivot)
industry_counts = (
    fac_matched.groupby(['PWSID', 'Industry'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
# Clean up column names
industry_counts.columns = [
    'PWSID' if c == 'PWSID' else f'n_{c.lower().replace(" ", "_")}'
    for c in industry_counts.columns
]

fac_summary = fac_count.merge(industry_counts, on='PWSID', how='left')

print(f'PWS with at least one facility: {len(fac_summary)}')
print(fac_summary.head())

PWS with at least one facility: 204
       PWSID  total_facilities  n_airports  n_airports_(part_139)  \
0  NJ0102001                 8           4                      0   
1  NJ0104003                 2           0                      0   
2  NJ0107001                 1           0                      0   
3  NJ0112001                 3           0                      0   
4  NJ0113001                 5           1                      0   

   n_cement_mfg  n_chemical_mfg  n_cleaning_product_mfg  n_consumer_products  \
0             0               0                       0                    0   
1             0               0                       0                    0   
2             0               0                       0                    0   
3             0               1                       0                    0   
4             0               0                       0                    0   

   n_electronics_industry  n_fire_protection  ...  n_mining_and_refi

## Merging Everything Together

In [ ]:
# Start with PWS boundaries as the base
proximity_df = pws_gdf[['PWSID', 'PWS_Name', 'Population_Served_Count', 'geometry']].copy()

# Join PFAS levels
proximity_df = proximity_df.merge(pfas_by_pws, on='PWSID', how='left')

# Join facility counts (fill 0 for PWS with no facilities)
proximity_df = proximity_df.merge(fac_summary, on='PWSID', how='left')
proximity_df['total_facilities'] = proximity_df['total_facilities'].fillna(0).astype(int)
proximity_df['has_facility'] = proximity_df['total_facilities'] > 0

# Join demographics via ZIP → PWSID crosswalk
zip_demo = zip_pws.merge(census_df, on='ZIPCODE', how='left')

# Convert demographic columns to numeric before aggregating
demo_cols = ['pct_poc', 'pct_black', 'pct_hispanic', 'pct_nonhispanic_white',
             'hh_median_income', 'total_pop', 'zhvi']
for col in demo_cols:
    zip_demo[col] = pd.to_numeric(zip_demo[col], errors='coerce')

demo_by_pws = zip_demo.groupby('PWSID')[demo_cols].mean().reset_index()

proximity_df = proximity_df.merge(demo_by_pws, on='PWSID', how='left')

print(f'Final dataset: {len(proximity_df)} water systems')
print(proximity_df.columns.tolist())

## Saving

In [22]:
# Save GeoDataFrame (with geometry) for mapping in notebook 05
proximity_df.to_file(DATA_OUT / 'proximity_final.geojson', driver='GeoJSON')

# Save flat CSV (without geometry) for stats/charts in notebook 05
proximity_df.drop(columns='geometry').to_csv(
    DATA_OUT / 'proximity_final.csv', index=False
)

# Save the facility-level data with PWS assignments for mapping
facilities_in_pws.drop(columns='geometry').to_csv(
    DATA_OUT / 'echo_facilities_with_pws.csv', index=False
)

print('Saved:')
print('  proximity_final.geojson')
print('  proximity_final.csv')
print('  echo_facilities_with_pws.csv')

Saved:
  proximity_final.geojson
  proximity_final.csv
  echo_facilities_with_pws.csv
